In [1]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## ഘട്ടം 1: ഘടനാപരമായ ഔട്ട്പുട്ടുകൾക്കായി Pydantic മോഡലുകൾ നിർവചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകുന്ന **സ്കീമ** നിർവചിക്കുന്നു. Pydantic ഉപയോഗിച്ച് `response_format` ഉപയോഗിക്കുന്നത് ഉറപ്പു നൽകുന്നു:
- ✅ തരം-സുരക്ഷിത ഡാറ്റ എക്straction
- ✅ സ്വയം വാലിഡേഷൻ
- ✅ ഫ്രീ-ടെക്‌സ്റ് പ്രതികരണങ്ങളിൽ നിന്ന് പാഴ്വാക്ക് പാഴ്വാക്ക് പാഴ്വാക്ക് പാഴ്വാക്ക് പാഴ്വാക്ക് പാഴ്
- ✅ ഫീൽഡുകൾ അടിസ്ഥാനമാക്കിയുള്ള എളുപ്പമുള്ള ശീർഷ്.routing


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## Step 2: ഹോട്ടല്‍ ബുക്കിംഗ് ടൂളിനെ സൃഷ്ടിക്കുക

റൂം ലഭ്യമാണോ എന്ന് പരിശോധിക്കാന്‍ **availability_agent** കോള്‍ ചെയ്യുന്നത് ഈ ടൂൾ ആണു്. Python ഫംഗ്‌ഷനെ AI-കോള്ബിള്‍ ടൂളായി മാറ്റാന്‍ ഞങ്ങള്‍ `@ai_function` ഡെക്കറേറ്റർ ഉപയോഗിക്കുന്നു:
- Python ഫംഗ്‌ഷനെ AI-കോള്ബിള്‍ ടൂളായി മാറ്റുക
- LLM-നായി JSON സ്കീമ സ്വയം സൃഷ്ടിക്കുക
- പാരാമീറ്റർ വാലിഡേഷൻ കൈകാര്യം ചെയ്യുക
- ഏജന്‍സികൾ സ്വയം invoke ചെയ്യുന്നതിന് സാധ്യമാക്കുക

ഈ ഡെമോയിനായി:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → റൂമുകൾ ലഭ്യമാണ് ✅
- **മറ്റ് എല്ലാ നഗരങ്ങളും** → റൂമുകൾ ഇല്ല ❌


In [3]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## ഘട്ടം 3: റൂട്ടിങ്ങിനുള്ള വ്യവസ്ഥ ഫംഗ്ഷനുകൾ നിർവചിക്കുക

ഈ ഫംഗ്ഷനുകൾ ഏജന്റിന്റെ പ്രതികരണം പരിശോധിച്ച് വർക്ക്‌ഫ്ലോയിൽ ഏത് പാത തിരഞ്ഞെടുക്കേണ്ടതാണെന്ന് നിർണ്ണയിക്കുന്നു.

**പ്രധാന പാറ്റേൺ:**
1. സന്ദേശം `AgentExecutorResponse` ആണോ എന്ന് പരിശോധിക്കുക
2. ഘടിതമായ ഔട്ട്പുട്ട് (പൈഡാന്റിക് മോഡൽ) പാഴ്‌സ് ചെയ്യുക
3. റൂട്ടിങ്ങ് നിയന്ത്രിക്കാൻ `True` അല്ലെങ്കിൽ `False` മടക്കി നൽകുക

വരിപാടി ഈ വ്യവസ്ഥകൾ **എഡ്ജുകളിൽ** മൂല്യനിർണ്ണയം ചെയ്ത് അടുത്ത എക്സിക്യൂട്ടറെ വിളിക്കേണ്ടതോhendöl കണ്ടെത്തും.


In [4]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)


## ഘട്ടം 4: കസ്റ്റം പ്രദർശന എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

എക്സിക്യൂട്ടറുകൾ പരിവർത്തനങ്ങൾ നടത്തുകയോ സൈഡ് എഫക്റ്റുകൾ ഉണ്ടാക്കുകയോ ചെയ്യുന്ന വർക്ക്ഫ്ലോ ഘടകങ്ങളാണ്. അവസാന ഫലം പ്രദർശിപ്പിക്കുന്ന കസ്റ്റം എക്സിക്യൂട്ടർ സൃഷ്ടിക്കാൻ `@executor` ഡെക്കറേറ്റർ我们 ഉപയോഗിക്കുന്നു.

**പ്രധാന ആശയങ്ങൾ:**
- `@executor(id="...")` - ഒരു ഫംഗ്ഷനെ വർക്ക്ഫ്ലോ എക്സിക്യൂട്ടറായി രജിസ്റ്റർ ചെയ്യുന്നു
- `WorkflowContext[Never, str]` - ഇൻപുട്ട്/ഔട്ട്പുട്ടിനുള്ള ടൈപ്പ് സൂചനകൾ
- `ctx.yield_output(...)` - അവസാന വർക്ക്ഫ്ലോ ഫലം നൽകുന്നു


In [5]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

✅ display_result executor created with @executor decorator


## ഘട്ടം 5: പരിസ്ഥിതി വേരിയബിളുകൾ ലോഡ് ചെയ്യുക

LLM ക്ലയന്റ് കോൺഫിഗർ ചെയ്യുക. ഈ ഉദാഹരണം പണിയെന്നു:
- **GitHub മോഡലുകൾ** (GitHub ടോക്കണോടെയുള്ള സൌജന്യ തരം)
- **Azure OpenAI**
- **OpenAI**


In [6]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")

## Step 6: ഘടനാപരമായ ഔട്ട്പുട്ടുകളോടുകൂടി AI ഏജന്റ്സ് സൃഷ്ടിക്കുക

നാം **മൂന്ന് പ്രത്യേക ഏജന്റ്സുകൾ** സൃഷ്ടിക്കുന്നു, ഓരോതും ഒരു `AgentExecutor`-ൽ പാക്ക് ചെയ്തിരിക്കുന്നു:

1. **availability_agent** - ടൂളിന്റെ സഹായത്തോടെ ഹോട്ടലിന്റെ ലഭ്യത പരിശോധിക്കുന്നു
2. **alternative_agent** - ബാക്കിയില്ലാത്തപ്പോൾ പകരം നഗരങ്ങൾ നിർദ്ദേശിക്കുന്നു
3. **booking_agent** - മുറികൾ ലഭ്യമാകുമ്പോൾ ബുക്കിംഗ് പ്രോത്സാഹിപ്പിക്കുന്നു

**പ്രധാന ലക്‌ഷണങ്ങൾ:**
- `tools=[hotel_booking]` - ഏജന്റ് ടൂളിന് നൽകുന്നു
- `response_format=PydanticModel` - ഘടനാപരമായ JSON ഔട്ട്പുട്ട് നിർബന്ധിതമാക്കുന്നു
- `AgentExecutor(..., id="...")` - വർക്ക്‌ഫ്ലോ ഉപയോഗത്തിന് ഏജന്റ് വ്രാപ്പ് ചെയ്യുന്നു


In [7]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ഘട്ടം 7: നിബന്ധനാപൂർവമായ കോണുകളുള്ള വർക്ക്‌ഫ്ലോ നിർമ്മിക്കുക

ഇപ്പോൾ നാം `WorkflowBuilder` ഉപയോഗിച്ച് നിബന്ധനാപൂർവമായ റൂട്ടിംഗുമായി ഗ്രാഫ് നിർമ്മിക്കുന്നു:

**വർക്ക്‌ഫ്ലോ ഘടന:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**പ്രധാനമായ മെത്തഡുകൾ:**
- `.set_start_executor(...)` - എൻട്രി പോയിന്റ് സജ്ജമാക്കുന്നു
- `.add_edge(from, to, condition=...)` - നിബന്ധനാപൂർവമായ കോണം ചേർക്കുന്നു
- `.build()` - വർക്ക്‌ഫ്ലോ അവസാനിപ്പിക്കുന്നു


In [8]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## പടി 8: ടെസ്റ്റ് കേസ് 1 ഓടിക്കുക - നഗരത്തിൽ ലഭ്യത ഇല്ല (പാരിസ്)

**ലഭ്യത ഇല്ലാതിരിക്കുന്ന** പാത പരീശೀಲിക്കാൻ പാരിസിൽ ഹോട്ടലുകൾ ആവശ്യപ്പെടാം (നമ്മുടെ സിമുലേഷനിൽ അവിടെ മുറികൾ ഇല്ല).


In [9]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: പരീക്ഷണ കേസ് 2 - സിറ്റി WITH ലഭ്യത (സ്റ്റോക്ക്‌ഹോൾം)

ഇപ്പോൾ സിമുലേഷനിൽ മുറികൾ ഉള്ള സ്റ്റോക്ക്‌ഹോൾത്തിലെ (Stockholm) ഹോട്ടലുകൾ ആവശ്യപ്പെടുന്നതിനിലൂടെ **ലഭ്യത** മാർഗ്ഗം പരിശോധിക്കാം.


In [10]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## പ്രധാനപ്പെട്ട പാഠങ്ങൾവും അടുത്ത് കൈകോർത്ത് കാര്യങ്ങളും

### ✅ നിങ്ങൾ പഠിച്ചത്:

1. **WorkflowBuilder പാറ്റേൺ**
   - എൻട്രി പോയിന്റ് നിർവചിക്കാൻ `.set_start_executor()` ഉപയോഗിക്കുക
   - ശൈതാന്റ്ര routing നു `.add_edge(from, to, condition=...)` ഉപയോഗിക്കുക
   - workflow അന്തിമമാക്കാൻ `.build()` കോളുചെയ്യുക

2. **ശൈതാന്റ്ര റൂട്ടിംഗ്**
   - കൺഡീഷൻ ഫംഗ്ഷനുകൾ `AgentExecutorResponse` പരിശോധിക്കുന്നു
   - സ്ട്രക്ചേച്ചേർഡ് ഔട്ട്പുട്ടുകൾ പാഴ്സ് ചെയ്ത് റൂട്ടിംഗ് തീരുമാനങ്ങൾ എടുക്കുക
   - ഒരു എജ്‌ സജീവമാക്കാൻ `True` തിരികെ നൽകുക, അതില്ലാതെ `False`

3. **ടൂൾ ഇന്റഗ്രേഷൻ**
   - Python ഫംഗ്ഷനുകൾ AI ടൂളുകളാക്കി മാറ്റാൻ `@ai_function` ഉപയോഗിക്കുക
   - ഏജന്റുകൾ ആവശ്യപ്പെടുമ്പോൾ ടൂളുകൾ സ്വയം വിളിക്കും
   - ടൂളുകൾ JSON ഫോർമാറ്റിൽ ഡാറ്റ റിട്ടേൺ ചെയ്യും, ഇത് ഏജന്റുകൾ പാഴ്സു ചെയ്യും

4. **സ്ട്രക്ചേച്ചേർഡ് ഔട്ട്പുട്ടുകൾ**
   - ടൈപ്പ്-സേഫ് ഡാറ്റ എക്സ്ട്രാക്ഷനായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക
   - ഏജന്റുകൾ സൃഷ്ടിക്കുമ്പോൾ `response_format=MyModel` സെറ്റ് ചെയ്യുക
   - മറുപടികൾ `Model.model_validate_json()` ഉപയോഗിച്ച് പാഴ്സ് ചെയ്യുക

5. **കസ്റ്റം എക്സിക്യൂട്ടറുകൾ**
   - workflow ഘടകങ്ങൾ സൃഷ്ടിക്കാൻ `@executor(id="...")` ഉപയോഗിക്കുക
   - എക്സിക്യൂട്ടറുകൾ ഡാറ്റ മാറ്റം വരുത്തുകയോ സൈഡ് ഇഫക്റ്റുകൾ നടത്തുകയോ ചെയ്യും
   - workflow ഫലം പ്രദാനം ചെയ്യാൻ `ctx.yield_output()` ഉപയോഗിക്കുക

### 🚀 യാഥാർത്ഥ്യ പ്രശ്നങ്ങളിൽ പ്രയോഗങ്ങൾ:

- **ട്രാവൽ ബുക്കിംഗ്**: ലഭ്യത പരിശോധിക്കുക, ബദൽ നിർദ്ദേശങ്ങൾ നൽകുക, ഓപ്ഷനുകൾ താരതമ്യം ചെയ്യുക
- **കസ്റ്റമർ സർവീസ്**: പ്രശ്ന തരം, വികാരം, മുൻഗണനയുടെ അടിസ്ഥാനത്തിൽ റൂട്ടിംഗ്
- **ഇ-കോമേഴ്‌സ്**: ഇനങ്ങൾ നിലയിലുള്ള സ്ഥിതി പരിശോധിക്കുക, ബദൽ നിർദ്ദേശിക്കുക, ഓർഡറുകൾ പ്രോസസ് ചെയ്യുക
- **കൺറന്റ് മോഡറേഷൻ**: ടോക്സിസിറ്റി സ്കോറുകൾ, ഉപഭോക്തൃ ഫ്‌ലാഗുകൾ അടിസ്ഥാനത്തിൽ റൂട്ടിംഗ്
- **അപ്രൂവൽ വർക്ക്ഫ്ലോകൾ**: തുക, ഉപഭോക്തൃ റോളുകൾ, റിസ്ക് നിലയുടെ അടിസ്ഥാനത്തിൽ റൂട്ടിംഗ്
- **മൾട്ടി-സ്റ്റേജ് പ്രോസസ്സിംഗ്**: ഡാറ്റ ഗുണമേന്മ, പൂർണ്ണത അടിസ്ഥാനത്തിൽ റൂട്ടിംഗ്

### 📚 അടുത്ത് കൈകോർത്ത് കാര്യങ്ങൾ:

- കൂടുതൽ സങ്കീർണമായ ശൈതാന്റ്റ്ര കൺഡീഷനുകൾ ചേർക്കുക (ബഹുമുഖ മാനദണ്ഡങ്ങൾ)
- workflow സ്റ്റേറ്റ് മാനേജ്‌മെന്റ് ഉപയോഗിച്ച് ലൂപ്പുകൾ നടപ്പിലാക്കുക
- ആവർത്തിക്കുന്ന ഘടകങ്ങൾക്ക് ഉപ-വർക്ക്ഫ്ലോകൾ ചേർക്കുക
- യഥാർത്ഥ APIകൾ (ഹോട്ടൽ ബുക്കിംഗ്, ഇൻവെന്ററി സംവിധാനങ്ങൾ) ഇന്റഗ്രേറ്റ് ചെയ്യുക
- പിശക് കൈകാര്യം ചെയ്യലും ഫാൾബാക്ക് മാർഗങ്ങളും ചേർക്കുക
- workflow കൾക്ക് ഇൻബിൽട്ട് വിസ്വലൈസേഷൻ ടൂളുകൾ ഉപയോഗിച്ച് ദൃശ്യവൽക്കരണം നൽകുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**വിവരണം**:  
ഈ ദസ്താവേള AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. നാം കൃത്യതതിനായി പരിശ്രമിച്ചേശൂവെങ്കിലും, ഓട്ടോമാറ്റഡ് വിവർത്തനങ്ങളിൽ പരമാവധി പിശകുകൾ അല്ലെങ്കിൽ അകൃത്വങ്ങൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കൂ. അതിന്റെ ജന്മഭൂമിയിലെ പ്രഥമ ദസ്താവേള അധികൃത് ഉറവിടമായി പരിഗണിക്കണം. പ്രധാന വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മാനവ വിവർത്തനം നിർദ്ദേശിക്കപ്പെടുന്നു. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗത്തിൽ ഉണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റായ ധാരണകൾക്കും വ്യാഖ്യാനത്തിനും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
